# AI Support Ticket Triage 

 
## Final Flow

```text
Subject + Body
      ↓
ML Model 1 → Issue Type
ML Model 2 → Priority
      ↓
Llama 3 via Ollama
      ↓
Queue + Summary + Main Problem + Recommended Action + Suggested Response
```

### Responsibility Split

**Machine Learning**
- Predict Issue Type
- Predict Priority

**Llama 3**
- Select the most appropriate Queue from a fixed allowed list
- Summarize the ticket
- Identify the main problem
- Recommend the next action
 

In [5]:
 # 2. IMPORTS
 
import json
import re
import requests
import pandas as pd
import joblib

In [11]:

# 3. CONFIGURATION
 
ISSUE_TYPE_MODEL_PATH = "../Downloads/sam/issue_type_model.joblib"
PRIORITY_MODEL_PATH = "../Downloads/sam/priority_model.joblib"

LLAMA_MODEL = "llama3"
OLLAMA_ENDPOINT = "http://localhost:11434/api/generate"

REQUEST_TIMEOUT = 120
TEMPERATURE = 0.2

# IMPORTANT:
ALLOWED_QUEUES = [
    "Billing and Payments",
    "Returns and Exchanges",
    "Technical Support",
    "Account Management",
    "General Inquiry"
]

## 4. Check Ollama Connection

In [7]:
# 4. CHECK OLLAMA


def check_ollama():
    try:
        response = requests.get(
            "http://localhost:11434/api/tags",
            timeout=10
        )

        response.raise_for_status()

        data = response.json()

        models = [
            model.get("name", "")
            for model in data.get("models", [])
        ]

        print("Ollama is running.")
        print("Available models:", models)

        if not any(
            model_name.startswith(LLAMA_MODEL)
            for model_name in models
        ):
            print(
                f"Warning: '{LLAMA_MODEL}' was not found. "
                f"Run: ollama pull {LLAMA_MODEL}"
            )

        return True

    except requests.RequestException as e:
        print("Ollama connection failed.")
        print("Make sure Ollama is running.")
        print("Error:", e)

        return False


check_ollama()

Ollama is running.
Available models: ['llama3:latest']


True

## 5. Load the Two ML Models

In [12]:
 # 5. LOAD TRAINED ML MODELS
 
issue_type_model = joblib.load(
    ISSUE_TYPE_MODEL_PATH
)

priority_model = joblib.load(
    PRIORITY_MODEL_PATH
)

print("Issue Type model loaded successfully.")
print("Priority model loaded successfully.")

Issue Type model loaded successfully.
Priority model loaded successfully.


## 6. Predict Issue Type and Priority



 

In [13]:
 # 6. ML PREDICTIONS
 
def predict_ticket_labels(subject, body):

    subject = "" if pd.isna(subject) else str(subject).strip()
    body = "" if pd.isna(body) else str(body).strip()

    ticket_text = f"{subject} {body}".strip()

    if not ticket_text:
        raise ValueError(
            "Ticket text cannot be empty."
        )

    predicted_type = issue_type_model.predict(
        [ticket_text]
    )[0]

    predicted_priority = priority_model.predict(
        [ticket_text]
    )[0]

    return {
        "ticket_text": ticket_text,
        "predicted_type": str(predicted_type),
        "predicted_priority": str(predicted_priority)
    }

## 7. Build the Llama Prompt

Llama receives:

- original subject
- original body
- ML-predicted Issue Type
- ML-predicted Priority
- fixed list of allowed Queues

Llama must choose exactly one Queue from that list.

In [14]:
 # 7. PROMPT BUILDER
 
def build_llama_prompt(
    subject,
    body,
    predicted_type,
    predicted_priority
):

    allowed_queues_text = "\n".join(
        f"- {queue}"
        for queue in ALLOWED_QUEUES
    )

    prompt = f"""
You are an AI assistant for a customer support ticket triage system.

The Machine Learning models have already classified the ticket's
Issue Type and Priority.

CUSTOMER TICKET

Subject:
{subject}

Body:
{body}


ML PREDICTIONS

Issue Type: {predicted_type}
Priority: {predicted_priority}


ALLOWED QUEUES

Choose exactly ONE Queue from the following list:

{allowed_queues_text}


RULES

1. Do not change the Issue Type.
2. Do not change the Priority.
3. Choose exactly one Queue from the allowed list.
4. Do not create a new Queue.
5. Select the Queue based on the ticket content and ML predictions.
6. Do not invent facts not present in the ticket.
7. Do not claim the issue has already been resolved.
8. Do not promise a resolution time unless explicitly stated.
9. Keep the output concise and professional.
10. Return valid JSON only.
11. Do not include markdown or explanation outside the JSON.

Return exactly this JSON structure:

{{
  "predicted_queue": "one exact Queue name from the allowed list",
  "summary": "a concise 1-2 sentence summary",
  "main_problem": "the main customer problem",
  "recommended_action": "the recommended next action for the selected support team",
  "suggested_response": "a short professional response to the customer"
}}
"""

    return prompt.strip()

## 8. Send the Prompt to Llama 3 Through Ollama

In [15]:
 # 8. LLAMA GENERATION THROUGH OLLAMA
 
def generate_with_llama(prompt):

    payload = {
        "model": LLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {
            "temperature": TEMPERATURE
        }
    }

    response = requests.post(
        OLLAMA_ENDPOINT,
        json=payload,
        timeout=REQUEST_TIMEOUT
    )

    response.raise_for_status()

    result = response.json()

    if "response" not in result:
        raise ValueError(
            "Unexpected Ollama response format."
        )

    return result["response"]

## 9. Parse and Validate the Llama Output

This validation checks that:

- all required GenAI fields exist
- the returned Queue is one of the allowed Queue names

In [16]:
 # 9. JSON PARSER AND VALIDATOR
 
REQUIRED_GENAI_FIELDS = [
    "predicted_queue",
    "summary",
    "main_problem",
    "recommended_action",
    "suggested_response"
]


def parse_llama_json(raw_output):

    if not isinstance(raw_output, str):
        raise TypeError(
            "Llama output must be a string."
        )

    cleaned = raw_output.strip()

    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned
    )

    first_brace = cleaned.find("{")
    last_brace = cleaned.rfind("}")

    if first_brace == -1 or last_brace == -1:
        raise ValueError(
            "No JSON object found in Llama output."
        )

    parsed = json.loads(
        cleaned[first_brace:last_brace + 1]
    )

    missing_fields = [
        field
        for field in REQUIRED_GENAI_FIELDS
        if field not in parsed
    ]

    if missing_fields:
        raise ValueError(
            f"Missing fields: {missing_fields}"
        )

    parsed_queue = str(
        parsed["predicted_queue"]
    ).strip()

    if parsed_queue not in ALLOWED_QUEUES:
        raise ValueError(
            f"Invalid queue returned by Llama: {parsed_queue}"
        )

    return {
        field: str(parsed[field]).strip()
        for field in REQUIRED_GENAI_FIELDS
    }

## 10. Complete End-to-End Pipeline

In [17]:
 # 10. COMPLETE PIPELINE
 
def process_ticket(subject, body):

    # Step 1: ML predictions
    ml_result = predict_ticket_labels(
        subject,
        body
    )

    # Step 2: Build Llama prompt
    prompt = build_llama_prompt(
        subject=subject,
        body=body,
        predicted_type=ml_result[
            "predicted_type"
        ],
        predicted_priority=ml_result[
            "predicted_priority"
        ]
    )

    # Step 3: Llama generation
    raw_llama_output = generate_with_llama(
        prompt
    )

    # Step 4: Parse and validate output
    genai_result = parse_llama_json(
        raw_llama_output
    )

    # Step 5: Final combined result
    return {
        "subject": subject,
        "body": body,
        "predicted_type": ml_result[
            "predicted_type"
        ],
        "predicted_priority": ml_result[
            "predicted_priority"
        ],
        **genai_result
    }

## 11. Test the ML Part Only

 

In [26]:
 # 11. TEST ML ONLY
 
example_subject = "Incorrect invoice amount"

example_body = (
    "Urgent The amount shown on my latest invoice is Wrong. "
    "Please review the charges."
)

ml_result = predict_ticket_labels(
    example_subject,
    example_body
)

print(
    json.dumps(
        ml_result,
        indent=2,
        ensure_ascii=False
    )
)

{
  "ticket_text": "Incorrect invoice amount Urgent The amount shown on my latest invoice is Wrong. Please review the charges.",
  "predicted_type": "incident",
  "predicted_priority": "low"
}


In [28]:
test_tickets = [
    "Urgent critical system outage. All services are down.",
    "Severe security incident. Unauthorized access detected.",
    "Please update my account information.",
    "I have a small question about my invoice.",
    "Password reset request.",
]

for ticket in test_tickets:
    pred = priority_model.predict([ticket])[0]
    print(f"{pred:8} | {ticket}")

high     | Urgent critical system outage. All services are down.
high     | Severe security incident. Unauthorized access detected.
high     | Please update my account information.
medium   | I have a small question about my invoice.
high     | Password reset request.


## 12. Test the Complete ML + GenAI System

Run this after confirming Ollama is running.

In [27]:
#  12. COMPLETE EXAMPLE



result = process_ticket(
    example_subject,
    example_body
)

print(
    json.dumps(
        result,
        indent=2,
        ensure_ascii=False
    )
)

{
  "subject": "Incorrect invoice amount",
  "body": "Urgent The amount shown on my latest invoice is Wrong. Please review the charges.",
  "predicted_type": "incident",
  "predicted_priority": "low",
  "predicted_queue": "Billing and Payments",
  "summary": "Invoice amount discrepancy reported.",
  "main_problem": "Incorrect invoice amount shown on latest invoice.",
  "recommended_action": "Verify invoice charges and investigate cause of discrepancy.",
  "suggested_response": "Thank you for bringing this to our attention. We will review the charges and get back to you shortly."
}
